## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Emoji reactions to posts: correspondence analysis of counts

**Correspondence Analysis** · a worked example from the StatNex teaching guide

**The story.** A brand's social-media team counted the emoji reactions (Like, Love, Haha, Wow, Sad, Angry) to 11 types of post over a year, 2,221 reactions in total.

**The question.** Which kinds of post trigger which emotions?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/correspondence_analysis__counts.csv` (12 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Title,Post type,Like,Love,Haha,Wow,Sad,Angry,Total
Post type,Product launch,120.0,40.0,8.0,65.0,2.0,6.0,241.0
,Behind the scenes,85.0,52.0,20.0,18.0,1.0,2.0,178.0
,Customer story,70.0,88.0,6.0,12.0,15.0,1.0,192.0
,Meme,60.0,25.0,140.0,10.0,2.0,4.0,241.0
,Discount offer,150.0,35.0,5.0,30.0,1.0,10.0,231.0
,Price increase,20.0,2.0,12.0,15.0,30.0,110.0,189.0
,Charity event,55.0,120.0,2.0,20.0,10.0,2.0,209.0
,Tutorial,95.0,30.0,10.0,25.0,1.0,3.0,164.0
,Giveaway,130.0,60.0,15.0,22.0,3.0,8.0,238.0
,Service outage,10.0,1.0,20.0,8.0,45.0,90.0,174.0
,Holiday greeting,75.0,70.0,10.0,5.0,3.0,1.0,164.0
,Total,870.0,523.0,248.0,230.0,113.0,237.0,2221.0
""")

## Helper: totals

A step that no library does, written out. In the code download it is the file `totals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# A cross tabulation of counts: one category of the row variable per row, one of the
# column variable per column.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Like", "Love", "Haha", "Wow", "Sad", "Angry"]
table = data.set_index("Post type")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
# is_total: defined in the helper cell totals above (totals.py in the code download)

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank cell is left out.
table = table.dropna()
counts = table.to_numpy(float)
n_rows, n_columns = counts.shape
row_labels, column_labels = list(table.index), list(table.columns)

## The table scaled to proportions

In [ ]:
# The table is divided by its grand total, so it sums to 1.
P = counts / counts.sum()
row_weight = P.sum(axis=1)
row_margin, column_margin = P.sum(axis=1), P.sum(axis=0)
row_mass, column_mass = row_margin / P.sum(), column_margin / P.sum()
column_weight = column_mass

## Cross tabulation

In [ ]:
grid = np.column_stack([counts, counts.sum(axis=1)])
grid = np.vstack([grid, grid.sum(axis=0)])
crosstab = pd.DataFrame(grid, index=[*row_labels, "Total"], columns=[*column_labels, "Total"])
print(crosstab)

## Row and column profiles

In [ ]:
# Each row (column) as shares of its own total. Correspondence analysis compares these:
# two rows are close on the map when their profiles are alike, whatever their totals.
row_profile = pd.DataFrame(np.vstack([counts / counts.sum(axis=1, keepdims=True), column_mass]),
                           index=[*row_labels, "Mass"], columns=column_labels)
column_profile = pd.DataFrame(np.column_stack([counts / counts.sum(axis=0), row_mass]),
                              index=row_labels, columns=[*column_labels, "Mass"])
print()
print(row_profile.round(4))
print()
print(column_profile.round(4))

## Decomposition

In [ ]:
# The standardised residuals from independence, (p_ij - r_i c_j) / sqrt(r_i c_j), are
# split by a singular value decomposition. A table with R rows and C columns has
# min(R, C) - 1 axes; an axis with no inertia at all is dropped.
expected = np.outer(row_margin, column_margin) / P.sum()
residual = (P - expected) / np.sqrt(expected)
U, singular, Vt = np.linalg.svd(residual, full_matrices=False)
keep = min(n_rows, n_columns) - 1
U, singular, V = U[:, :keep], singular[:keep], Vt[:keep].T
live = singular > 1e-12 * max(1.0, singular[0])
U, singular, V = U[:, live], singular[live], V[:, live]
# An eigenvector and its negative describe the same axis, so its direction is a
# convention. The app's: flip each axis so its largest row-side component (in absolute
# value) is positive. Another program may draw the map mirrored, and when two
# components tie for the largest (two rows of equal weight, say) rounding decides.
for k in range(len(singular)):
    if U[np.argmax(np.abs(U[:, k])), k] < 0:
        U[:, k], V[:, k] = -U[:, k], -V[:, k]
eigen = singular**2
share = eigen / eigen.sum()
axes = [f"Axis {k + 1}" for k in range(len(singular))]

## Eigenvalues

In [ ]:
# The eigenvalue (the singular value squared) is the inertia the axis carries; its share
# of the total decides how many axes are worth reading.
eigenvalues = pd.DataFrame({"singular value": singular, "eigenvalue": eigen, "share": share,
                            "cumulative share": share.cumsum()}, index=axes)
print()
print(eigenvalues)

## Tests of the axes

In [ ]:
# Bartlett's statistic, the one canonical correlation uses, with the eigenvalue as the
# squared correlation: chi-square = -[n - (R + C + 1) / 2] ln(1 - eigenvalue) on
# R + C - 2k - 1 degrees of freedom for axis k. The residual test adds up the axes from
# m on, on (R - m)(C - m) degrees of freedom: is there still structure after m - 1 axes?
n = counts.sum()
factor = n - (n_rows + n_columns + 1) / 2
# An eigenvalue of exactly 1 (perfect association) is capped just below 1, so the
# logarithm stays finite.
log_terms = np.log1p(-np.minimum(eigen, 1 - 1e-12))
axis_chi2 = -factor * log_terms
axis_df = n_rows + n_columns - 2 * np.arange(1, len(eigen) + 1) - 1
axis_test = pd.DataFrame({"chi-square": axis_chi2, "df": axis_df,
                          "P": stats.chi2.sf(axis_chi2, axis_df)}, index=axes)
print()
print(axis_test)
tails = np.arange(1, len(eigen))
residual_chi2 = np.array([-factor * log_terms[m:].sum() for m in tails])
residual_df = (n_rows - tails - 1) * (n_columns - tails - 1)
residual_names = [f"Axis {m + 1} and below" for m in tails[:-1]] + axes[-1:]
residual_test = pd.DataFrame({"chi-square": residual_chi2, "df": residual_df,
                              "P": stats.chi2.sf(residual_chi2, residual_df)},
                             index=residual_names[:len(tails)])
print()
print(residual_test)

## Row and column summaries

In [ ]:
# For each level and each axis:
#   eigenvector                    the singular vector itself
#   score                          eigenvector / sqrt(weight) * sqrt(singular value): the
#                                  symmetric scaling that puts rows and columns on one map
#   contribution to the axis       eigenvector^2: how much of the axis the level makes
#   contribution within the level  the share of the level's inertia the axis explains
# Mass is the level's share of the table; inertia its share of the total inertia.
def summary(labels, vectors, scores, mass):
    parts = vectors**2 * eigen
    level_totals = parts.sum(axis=1, keepdims=True)
    # A level with no inertia (a profile equal to the average one, such as a Total row)
    # gets 0 rather than 0 / 0. Its inertia is zero only up to rounding, so "zero" means
    # below 1e-12 of the total.
    zero = level_totals <= 1e-12 * eigen.sum()
    within_level = np.divide(parts, level_totals, out=np.zeros_like(parts), where=~zero)
    groups = {"Eigenvector": vectors, "Score": scores, "Contribution to the axis": vectors**2,
              "Contribution within the level": within_level}
    out = pd.DataFrame(np.hstack(list(groups.values())), index=labels,
                       columns=[f"{group} {axis}" for group in groups for axis in axes])
    out.insert(0, "Inertia", parts.sum(axis=1) / eigen.sum())
    out.insert(0, "Mass", mass)
    return out


row_scores = U / np.sqrt(row_weight)[:, None] * np.sqrt(singular)
column_scores = V / np.sqrt(column_weight)[:, None] * np.sqrt(singular)
row_summary = summary(row_labels, U, row_scores, row_mass)
column_summary = summary(column_labels, V, column_scores, column_mass)
print()
print(row_summary.round(4).T)
print()
print(column_summary.round(4).T)

## Scores

In [ ]:
# The coordinates of the map: rows and columns on the same axes. A row and a column that
# lie in the same direction from the centre occur together more often than independence
# would predict; the exact distance between a row point and a column point means little.
scores = pd.DataFrame(np.vstack([row_scores, column_scores]), index=[*row_labels, *column_labels],
                      columns=axes)
print()
print(scores.round(4))

## What the result says

Correspondence analysis draws a map of a table of counts: rows and columns that go together get similar scores on an axis. The share of an axis says how much of the pattern it shows. Axis 1 has a share of 56.6% and axis 2 has 28.8%, so two axes show 85.3% of the pattern.

On axis 1, Price increase (1.90) and Service outage (1.97) sit far out with Angry (2.12) and Sad (1.55). On axis 2, Meme (2.05) sits with Haha (2.04). All other post types have negative scores on axis 1, near Like and Love. In the row profiles, 57% of reactions to charity events are Love, and 27% of reactions to product launches are Wow, against 10% for all posts.

So the posts trigger three clear patterns: anger, laughter and warmth.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.